# Day 11 — Debugging, logging, profiling
Objectives:
- Use print vs logging.
- Basic profiling with timeit/cProfile.
- Identify hot spots and optimize.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-11`. Read
`python/ds-60day/companion-guides/day11_debug_logging_profiling.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

Debugging is the process of reducing uncertainty. Start from an
observable symptom, make the smallest reproducible case, form one
hypothesis, and collect evidence that could disprove it. Reading the
traceback from the final exception line upward usually identifies the
failure type before the stack frames identify how execution arrived
there.

Logging records meaningful runtime events for later inspection;
debugging explores a live failure; profiling measures where time or
memory is actually spent. A log entry should add context without
exposing credentials or personal data. Optimization begins after a
representative measurement, not after guessing which line “looks slow.”

### Vocabulary

- **symptom:** the observable incorrect output, failure, or slowdown.
- **traceback:** the exception report showing failure type and active call stack.
- **hypothesis:** a testable explanation for the observed symptom.
- **log level:** severity such as DEBUG, INFO, WARNING, or ERROR.
- **profiler:** a tool that measures resource use by code location.
- **benchmark:** a repeatable timing or resource comparison under stated conditions.

## Syntax anatomy

`logger.info("loaded rows=%d", row_count)` separates a stable message
template from its value; logging formats it only when the level is
enabled. In a traceback, the last line names the exception, while the
preceding frames show calls from outermost to innermost. With `timeit`,
setup and repeated statement execution are deliberately separated.

### Worked example 1 — Add context at the boundary

A small function logs an outcome without dumping the input records. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import logging

logger = logging.getLogger("day11")
logger.setLevel(logging.INFO)

def accepted_count(values: list[int]) -> int:
    count = sum(value >= 0 for value in values)
    logger.info("accepted=%d total=%d", count, len(values))
    return count

accepted_count([4, -1, 0])

**Expected observation:** The function returns `2`; when the notebook has a visible logging handler, one INFO event reports counts rather than raw sensitive values.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Measure two equivalent membership strategies

Use repeated representative work instead of one noisy timestamp. Predict first; then run the next cell.

In [ ]:
from timeit import timeit

values = list(range(1_000))
value_set = set(values)
list_seconds = timeit("999 in values", number=5_000, globals=globals())
set_seconds = timeit("999 in value_set", number=5_000, globals=globals())
{"same_answer": (999 in values) == (999 in value_set),
 "set_faster_here": set_seconds < list_seconds}

**Expected observation:** Both lookups agree and the set is normally faster in this repeated lookup scenario. Exact durations vary by computer and are not asserted.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Write the exact expected and actual behavior before changing code.
2. Reduce the input until the failure still reproduces, then inspect the final traceback line.
3. Add structured context such as IDs or counts, never passwords or entire private records.
4. Benchmark the original and candidate with the same setup and verify they return equivalent results.

**Alternative to compare:** A debugger is best for stepping through changing state; temporary prints can help a tiny example; logging is better for repeatable or production execution.

**Boundary to test:** Nondeterminism, first-run cache warm-up, logging duplicate handlers, swallowed exceptions, and unrepresentative benchmark data can mislead diagnosis.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import logging, timeit
logging.basicConfig(level=logging.INFO)

def slow_sum(n: int) -> int:
    s = 0
    for i in range(n):
        s += i
    return s

logging.info('Running timeit...')
timeit.timeit(lambda: slow_sum(10_000), number=10)


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Start from a supplied failing function and use `breakpoint()` or the VS Code debugger to pause immediately before the wrong value is produced. **Evidence:** record the call arguments, two relevant local variables, and the branch taken. **Constraint:** do not change logic until you can state one falsifiable hypothesis.
   **Verify:** record the original failing test result before repair, then assert that exact input and one nearby passing input both produce their expected values after the fix.

2. Add module-level logging with `logging.getLogger(__name__)` and emit useful DEBUG/INFO/WARNING events for a small processing function. **Constraints:** use lazy `%s`/`%d` formatting, do not call `basicConfig` inside reusable library logic, and log counts/identifiers rather than sensitive record contents.
   **Verify:** demonstrate that changing the configured level changes visibility without changing the returned value.

3. Profile a deliberately slow membership or aggregation function with `cProfile` or `timeit`, implement one behavior-preserving improvement, and compare under identical inputs.
   **Expected behavior:** outputs match exactly and the measurement identifies where time changed. **Constraint:** report repeated timings rather than claiming from a single run.
   **Verify:** Assert old and new functions return identical results, then report repeated measurements and the profiler line/call count supporting the change.

### Additional mastery practice

Observe before optimizing: reproduce a defect, add bounded context, profile representative work, and change the measured bottleneck.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

4. **Prediction:** With a logger set to `WARNING`, predict which of DEBUG, INFO, WARNING, and ERROR calls are emitted.
   **Progressive hint:** The threshold keeps records at that level or more severe.
   **Verify:** Capture emitted records and assert WARNING and ERROR appear while DEBUG and INFO do not at a WARNING threshold.
5. **Tracing:** Trace a nested call failure and identify the first frame you own, the input value, and the violated assumption.
   **Progressive hint:** Read a traceback from the final exception upward through your code.
   **Verify:** Annotate the traceback with failure type, first owned frame, input, and violated assumption; rerun the minimal fixture to reproduce it exactly.
6. **Implementation:** Implement a reusable `timed(label)` context manager using `time.perf_counter` and logging.
   **Progressive hint:** Put elapsed-time logging in `finally` so failures are still timed.
   **Verify:** Capture one success and one raised block; assert both log a non-negative elapsed duration and the original exception is not swallowed.
7. **Debugging:** Explain why repeated `logging.basicConfig(...)` calls in notebooks may appear ineffective and configure a named logger without duplicate handlers.
   **Progressive hint:** Configuration is process state; inspect handlers before adding one.
   **Verify:** Run configuration twice and assert the named logger has exactly one intended handler and emits one copy of each message.
8. **Edge case and explanation:** Design a fair comparison between a loop and an alternative: include warm-up, equal inputs, repeated trials, and result verification.
   **Progressive hint:** A faster wrong answer is not an optimization.
   **Verify:** Assert both implementations return identical values across all benchmark inputs, then report warm-up and multiple comparable trial distributions.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Start from a supplied failing function and use `breakpoint()` or the VS Code debugger to pause immediately before the wrong value is produced. **Evidence:** record the call arguments, two relevant local variables, and the branch taken. **Constraint:** do not change logic until you can state one falsifiable hypothesis. **Verify:** record the original failing test result before repair, then assert that exact input and one nearby passing input both produce their expected values after the fix.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Start from a supplied failing function and use `breakpoint()` or the VS Code debugger to pause immediately before the wrong value is produced. record the call arguments, two rel...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Add module-level logging with `logging.getLogger(__name__)` and emit useful DEBUG/INFO/WARNING events for a small processing function. **Constraints:** use lazy `%s`/`%d` formatting, do not call `basicConfig` inside reusable library logic, and log counts/identifiers rather than sensitive record contents. **Verify:** demonstrate that changing the configured level changes visibility without changing the returned value.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Add module-level logging with `logging.getLogger(__name__)` and emit useful DEBUG/INFO/WARNING events for a small processing function. use lazy `%s`/`%d` formatting, do not call...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** Profile a deliberately slow membership or aggregation function with `cProfile` or `timeit`, implement one behavior-preserving improvement, and compare under identical inputs. **Expected behavior:** outputs match exactly and the measurement identifies where time changed. **Constraint:** report repeated timings rather than claiming from a single run. **Verify:** Assert old and new functions return identical results, then report repeated measurements and the profiler line/call count supporting the change.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Profile a deliberately slow membership or aggregation function with `cProfile` or `timeit`, implement one behavior-preserving improvement, and compare under identical inputs. ou...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** With a logger set to `WARNING`, predict which of DEBUG, INFO, WARNING, and ERROR calls are emitted. **Progressive hint:** The threshold keeps records at that level or more severe. **Verify:** Capture emitted records and assert WARNING and ERROR appear while DEBUG and INFO do not at a WARNING threshold.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: With a logger set to `WARNING`, predict which of DEBUG, INFO, WARNING, and ERROR calls are emitted. The threshold keeps records at that level or more severe. Capture emitted rec...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace a nested call failure and identify the first frame you own, the input value, and the violated assumption. **Progressive hint:** Read a traceback from the final exception upward through your code. **Verify:** Annotate the traceback with failure type, first owned frame, input, and violated assumption; rerun the minimal fixture to reproduce it exactly.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Trace a nested call failure and identify the first frame you own, the input value, and the violated assumption. Read a traceback from the final exception upward through your cod...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement a reusable `timed(label)` context manager using `time.perf_counter` and logging. **Progressive hint:** Put elapsed-time logging in `finally` so failures are still timed. **Verify:** Capture one success and one raised block; assert both log a non-negative elapsed duration and the original exception is not swallowed.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Implement a reusable `timed(label)` context manager using `time.perf_counter` and logging. Put elapsed-time logging in `finally` so failures are still timed. Capture one success...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Explain why repeated `logging.basicConfig(...)` calls in notebooks may appear ineffective and configure a named logger without duplicate handlers. **Progressive hint:** Configuration is process state; inspect handlers before adding one. **Verify:** Run configuration twice and assert the named logger has exactly one intended handler and emits one copy of each message.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Explain why repeated `logging.basicConfig(...)` calls in notebooks may appear ineffective and configure a named logger without duplicate handlers. Configuration is process state...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 8 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Design a fair comparison between a loop and an alternative: include warm-up, equal inputs, repeated trials, and result verification. **Progressive hint:** A faster wrong answer is not an optimization. **Verify:** Assert both implementations return identical values across all benchmark inputs, then report warm-up and multiple comparable trial distributions.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 8 — your work
# Short contract: Design a fair comparison between a loop and an alternative: include warm-up, equal inputs, repeated trials, and result verification. A faster wrong answer is not an optimization...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
